# Campaign-20 test-fragment inference viewer

Loads the exact architecture and inference geometry from run `20_bce_soft_noweight_5090_09_20-59-17`, then renders each test fragment and the w055 holdout.

Each full-size figure contains:

| raw inference | TTA inference | fiber composite | fiber composite with scale bar |

- the checkpoint is loaded strictly after removing training-only heads
- inference uses the shared bounded-memory row reader
- the visualizer module is reloaded so stale notebook kernels cannot retain an older implementation
- context, depth, multitile, TTA, and model settings come from the saved run config
- composites are cached separately because they are expensive to generate
- set `DRY_RUN = True` in cell 2 to inspect composites without loading the model

In [ ]:
# campaign-23 literal-surface slice-8 run
import json
import os
from pathlib import Path

DRY_RUN = False
EXP_NAME = os.getenv("VESUVIUS_EXP_NAME", "23_baseline")
RUN_ROOT = Path(os.getenv("VESUVIUS_RUN_ROOT", "/vesuvius/runs_archs23"))
RUN_ID = os.getenv("VESUVIUS_RUN_ID", "")
if RUN_ID:
    RUN_DIR = RUN_ROOT / RUN_ID
else:
    candidates = sorted(RUN_ROOT.glob(f"{EXP_NAME}_*/config.json"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"no completed run config matching {EXP_NAME!r} under {RUN_ROOT}")
    RUN_DIR = candidates[-1].parent
    RUN_ID = RUN_DIR.name
RUN_CONFIG_PATH = RUN_DIR / "config.json"
MODEL_PATH = Path(os.getenv(
    "VESUVIUS_MODEL_PATH",
    "/vesuvius/models/archs23/baseline_best_character.pth",
))
if not MODEL_PATH.exists():
    MODEL_PATH = Path("/vesuvius/models/archs23/baseline.pth")

with open(RUN_CONFIG_PATH, "r", encoding="utf-8") as f:
    RUN_CONFIG = json.load(f)
_RUN_DATA = RUN_CONFIG["data"]
_RUN_MODEL = RUN_CONFIG["model"]
ARCH = _RUN_MODEL["arch"]
TILE_SIZE = int(_RUN_DATA["tile_size"])
CONTEXT_SIZE = int(_RUN_DATA["context_size"])
CONTEXT_DOWNSAMPLE = int(_RUN_DATA["context_downsample"])
DEPTH = int(_RUN_DATA["depth"])
D_START = int(_RUN_DATA["d_start"])
D_END = int(_RUN_DATA["d_end"])
TTA = True
TTA_MODE = _RUN_DATA["tta_mode"]
INFER_BS = int(_RUN_DATA["eval_infer_bs"])
EVAL_PREFETCH = int(_RUN_DATA["eval_prefetch"])
EVAL_CHUNK_GB = float(_RUN_DATA["eval_chunk_gb"])
PRELOAD_RAM = True
TORCH_COMPILE = False
COMPOSITE_METHOD = "maxproj"
COMPOSITE_D0 = 10
COMPOSITE_D1 = 18
COMPOSITE_DISPLAY = "raw"
COMPOSITE_CLAHE = False
GENERATE_COMPOSITES = False
VOXEL_UM = 9.362
DISPLAY_SCALE = 0.25
MASK_CROP_MARGIN = 32
PAD_PX = 24
OUTPUT_DIR = "output"
COMPOSITE_CACHE = "output/composite_cache"
FRAGMENTS = {
    "auto_grown_20260814140748": 20260814140748,
    "auto_grown_20260717193517": 20260717193517,
    "auto_grown_20260720090842": 20260720090842,
    "auto_grown_20250703034159": 20250703034159,
    "auto_grown_20260723112922": 20260723112922,
}
if DEPTH != 8 or not _RUN_MODEL.get("surface_teacher_input") or not _RUN_DATA.get("surface_relative_depth_window"):
    raise RuntimeError("selected run is not literal-surface true slice-8")
print(RUN_ID, MODEL_PATH, f"depth={DEPTH}", "literal surface", f"ctx={CONTEXT_SIZE}/ds{CONTEXT_DOWNSAMPLE}", f"test_scrolls={len(FRAGMENTS)}")

20_bce_soft_noweight_5090_09_20-59-17 | nnunet3d_lcndz | tile=16 | ctx=192/ds2 | center=64


In [7]:
import gc
import importlib
import os
import sys
import types

REPO = os.getenv("VESUVIUS_REPO", "/vesuvius" if os.name == "posix" else r"C:\Users\ChenJeff\Documents\vesuvius")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

from utils.config import Config
from utils.dataloader import _load_unified_cache
from utils.model import create_model
from utils import visualizer as visualizer_module

# reload the bounded-memory row reader in existing kernels
visualizer_module = importlib.reload(visualizer_module)
TBV = visualizer_module.TensorboardVisualizer
group_by_depth = visualizer_module.group_by_depth
predict_tiles = visualizer_module.predict_tiles
load_or_create_midslice_mask = visualizer_module.load_or_create_midslice_mask

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(COMPOSITE_CACHE, exist_ok=True)
print(
    f"repo={REPO}",
    f"| torch={torch.__version__}",
    f"| cuda={torch.cuda.is_available()}",
    f"| dry_run={DRY_RUN}",
    "| bounded-memory visualizer reloaded",
)

repo=/vesuvius | torch=2.11.0+cu128 | cuda=True | dry_run=False | bounded-memory visualizer reloaded


In [ ]:
# ============ COMPOSITE (fiber-visibility) — reusable fn + method playground ============
# the composite is a fiber-visibility render straight from the zarr (NOT a prediction). it
# collapses a depth window into one 2D grayscale image so text predictions can later be
# overlaid on visible papyrus fibers. expensive on big fragments -> CACHED as PNG in
# output/composite_cache/. set GENERATE_COMPOSITES=True to force clean regeneration.
#
# MATCHED TO VC3D (core/util/Compositing.hpp): it applies a MAX filter over a LIMITED window
# (~8 layers around the surface, e.g. slices 10-18), NOT all 28 — deep layers add a dark band
# VC3D never shows. display is a linear volume window mapping raw 0-255 (COMPOSITE_DISPLAY="raw"),
# NOT a per-image percentile stretch or CLAHE (those manufacture contrast that isn't in VC3D).

def _project_depth(vol, d0, d1, method):
    """memory-bounded depth projection over slices [d0, d1) of a zarr volume (one slice at a time)."""
    H, W = int(vol.shape[1]), int(vol.shape[2])
    d0, d1 = max(0, d0), min(d1, int(vol.shape[0]))
    if method == "maxproj":
        acc = np.zeros((H, W), np.float32)
        for d in range(d0, d1):
            acc = np.maximum(acc, np.asarray(vol[d]).astype(np.float32))
        return acc
    if method == "meanproj":
        acc = np.zeros((H, W), np.float64)
        for d in range(d0, d1):
            acc += np.asarray(vol[d])
        return (acc / max(d1 - d0, 1)).astype(np.float32)
    if method == "minproj":
        acc = np.full((H, W), np.inf, np.float32)
        for d in range(d0, d1):
            acc = np.minimum(acc, np.asarray(vol[d]).astype(np.float32))
        return np.where(np.isfinite(acc), acc, 0).astype(np.float32)
    if method == "stdproj":   # depth-variance highlights fiber structure
        s = np.zeros((H, W), np.float64); s2 = np.zeros((H, W), np.float64)
        for d in range(d0, d1):
            sl = np.asarray(vol[d]).astype(np.float64); s += sl; s2 += sl * sl
        n = max(d1 - d0, 1); m = s / n
        return np.sqrt(np.clip(s2 / n - m * m, 0, None)).astype(np.float32)
    if method == "midslice":
        return np.asarray(vol[(d0 + d1) // 2]).astype(np.float32)
    raise ValueError(f"unknown composite method: {method}")

def _display_map(proj, m, mode, use_clahe):
    """map a float projection to uint8 for display.
    mode='raw'     -> direct 0-255 (VC3D's linear volume window; honest, low contrast).
    mode='stretch' -> robust 1-99 pct stretch computed WITHIN the mask (punchier)."""
    if mode == "raw":
        img = np.clip(proj, 0, 255).astype(np.uint8)
    else:
        lo, hi = np.percentile(proj[m], [1, 99]) if m.any() else (float(proj.min()), float(proj.max()))
        disp = np.clip((proj - lo) / max(hi - lo, 1e-6), 0, 1)
        img = (disp * 255).astype(np.uint8)
    if use_clahe:
        img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    return img * m.astype(np.uint8)   # zero outside the mask

def make_composite(scroll_id, method=None, d0=None, d1=None, use_cache=True, display=None, clahe=None):
    """return a uint8 fiber-visibility composite (2D) for a scroll, cached as PNG.
    depth window defaults to COMPOSITE_D0..COMPOSITE_D1 (VC3D's ~8-layer surface window).
    GENERATE_COMPOSITES=True forces recompute + overwrite regardless of use_cache."""
    method  = method or COMPOSITE_METHOD
    display = display or COMPOSITE_DISPLAY
    clahe   = COMPOSITE_CLAHE if clahe is None else clahe
    vol = zarr.open(os.path.join(Config().data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    Z = int(vol.shape[0])
    d0 = COMPOSITE_D0 if d0 is None else d0
    d1 = (COMPOSITE_D1 if COMPOSITE_D1 is not None else Z) if d1 is None else d1
    d0, d1 = max(0, d0), min(d1, Z)
    tag = f"_{display}" + ("_clahe" if clahe else "")
    cache_p = os.path.join(COMPOSITE_CACHE, f"{scroll_id}_{method}_{d0}-{d1}{tag}.png")
    if use_cache and not GENERATE_COMPOSITES and os.path.exists(cache_p):
        return np.array(Image.open(cache_p).convert("L"))
    mpath = f"masks/{scroll_id}.png"
    mask = (np.array(Image.open(mpath).convert("L")) > 0) if os.path.exists(mpath) else None
    proj = _project_depth(vol, d0, d1, method)
    m = mask if (mask is not None and mask.shape == proj.shape) else (proj > 0)
    img = _display_map(proj, m, display, clahe)
    Image.fromarray(img).save(cache_p)
    print(f"[composite] {scroll_id} {method} d{d0}-{d1}{tag} -> {cache_p}  {img.shape}"
          f"{'  (forced regen)' if GENERATE_COMPOSITES else ''}")
    return img

# --- playground: compare projection methods on the smallest test scroll ---
# VC3D match = maxproj + raw display over the COMPOSITE_D0..D1 surface window.
# (skipped on DRY_RUN; set DRY_RUN=False to experiment, then pick settings in the CONFIG cell)
# if not DRY_RUN:
#     PLAY_ID = 20260720090842   # PHerc1203 (smallest current test patch -> cheap to recompute)
#     trials = [("maxproj", "raw"), ("maxproj", "stretch"), ("meanproj", "raw"), ("midslice", "raw")]
#     fig, axes = plt.subplots(1, len(trials), figsize=(4 * len(trials), 4))
#     for ax, (meth, disp) in zip(axes, trials):
#         img = make_composite(PLAY_ID, method=meth, display=disp, use_cache=False, clahe=False)
#         ax.imshow(img, cmap="gray")
#         ax.set_title(f"{meth} {disp}"); ax.axis("off")
#     plt.tight_layout(); plt.show()
# else:
#     print("[playground] skipped on DRY_RUN")

In [ ]:
# inference and reusable figure builder
_AUXILIARY_PREFIXES = ("supcon_head.", "domain_head.")


def build_config():
    c = Config()
    for name in (
        "tile_size", "depth", "context_size", "context_downsample", "d_start", "d_end",
        "train_d_start", "train_d_end", "surface_label_dir", "surface_relative_depth_window",
        "eval_infer_bs", "eval_prefetch", "eval_chunk_gb", "tta_mode",
    ):
        if name in _RUN_DATA and hasattr(c.data, name):
            setattr(c.data, name, _RUN_DATA[name])
    for name, value in _RUN_MODEL.items():
        if hasattr(c.model, name):
            setattr(c.model, name, value)
    c.model.compile_model = False
    c.tra.supcon = False
    c.tra.dann = False
    c.device = "cuda" if torch.cuda.is_available() else "cpu"
    return c


def _checkpoint_state(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    cleaned = {}
    for key, value in state.items():
        key = key.removeprefix("module.").removeprefix("_orig_mod.")
        if not key.startswith(_AUXILIARY_PREFIXES):
            cleaned[key] = value
    return cleaned, len(state) - len(cleaned)


def load_model(c):
    model, n = create_model(c)
    state, ignored = _checkpoint_state(MODEL_PATH)
    model.load_state_dict(state, strict=True)
    model.eval()
    if TORCH_COMPILE:
        model = torch.compile(model, mode="reduce-overhead")
    print(f"loaded {MODEL_PATH} params={n:,} ignored_training_only={ignored}")
    return model


def get_norm(scroll_id, vol, mask):
    stats = _load_unified_cache().get(str(scroll_id))
    if stats and all(key in stats for key in ("mean", "std", "min", "max")):
        return stats["mean"], stats["std"], stats["min"], stats["max"]
    raise RuntimeError(f"normalization cache missing for {scroll_id}")


def predict_map(model, c, scroll_id):
    """predict raw and TTA maps using one literal-surface-centered eight-slice pass"""
    import time as _time
    vol = zarr.open(os.path.join(c.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    mask = np.array(Image.open(f"masks/{scroll_id}.png").convert("L")) > 0
    height = min(int(vol.shape[1]), int(mask.shape[0]))
    width = min(int(vol.shape[2]), int(mask.shape[1]))
    mask = mask[:height, :width]
    if PRELOAD_RAM:
        t0 = _time.time()
        print(f"[preload] {scroll_id} -> RAM ... ", end="", flush=True)
        vol = np.asarray(vol[:, :height, :width])
        print(f"done in {_time.time() - t0:.1f}s", flush=True)
    mean, std, global_min, global_max = get_norm(scroll_id, vol, mask)
    surface_dir = os.path.join(c.data.surface_label_dir, str(scroll_id))
    surface_depth = np.load(os.path.join(surface_dir, "depth.npy"), mmap_mode="r")
    surface_confidence = np.load(os.path.join(surface_dir, "confidence.npy"), mmap_mode="r")
    fake = types.SimpleNamespace(c=c)
    y_range, x_range = (0, height), (0, width)
    coords = TBV._gen_tile_coords(
        fake, (c.data.d_start, c.data.d_end), y_range, x_range, mask, z_step=c.data.depth
    )
    grouped = group_by_depth(coords)
    if len(grouped) != 1:
        raise RuntimeError(f"surface-relative inference expected one depth pass, got {len(grouped)}")
    depth_offset = next(iter(grouped))
    result = predict_tiles(
        c, model, vol, mask, grouped[depth_offset], y_range, x_range,
        c.data.d_start + depth_offset, f"test_{scroll_id}", mean, std, global_min, global_max,
        also_tta=TTA, surface_depth_map=surface_depth, surface_confidence_map=surface_confidence,
    )
    return result if TTA else (result, result)


def _mask_bbox(mask_bool, margin):
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return 0, mask_bool.shape[0], 0, mask_bool.shape[1]
    return (
        max(0, int(ys.min()) - margin), min(mask_bool.shape[0], int(ys.max()) + 1 + margin),
        max(0, int(xs.min()) - margin), min(mask_bool.shape[1], int(xs.max()) + 1 + margin),
    )


def _colorize(pmap, out_hw):
    valid = np.isfinite(pmap)
    p8 = (np.clip(np.nan_to_num(pmap, nan=0.0), 0, 1) * 255).astype(np.uint8)
    bgr = cv2.applyColorMap(p8, cv2.COLORMAP_INFERNO)
    bgr[~valid] = (115, 115, 115)
    return cv2.resize(bgr, (out_hw[1], out_hw[0]), interpolation=cv2.INTER_NEAREST)


def _red_scale_bar_1cm(bgr):
    height, width = bgr.shape[:2]
    length = int(round(10000.0 / VOXEL_UM))
    red = (0, 0, 255)
    thickness = max(4, min(height, width) // 250)
    pad = int(0.05 * min(height, width)) + thickness
    if width >= height:
        x1, y = width - pad, height - pad
        x0 = max(0, x1 - length)
        cv2.rectangle(bgr, (x0, y - thickness), (x1, y), red, -1)
        cv2.putText(bgr, "1 cm", (x0, y - thickness - 12), cv2.FONT_HERSHEY_SIMPLEX, 1.8, red, 3, cv2.LINE_AA)
    else:
        y1, x = height - pad, width - pad
        y0 = max(0, y1 - length)
        cv2.rectangle(bgr, (x - thickness, y0), (x, y1), red, -1)
        cv2.putText(bgr, "1 cm", (max(0, x - 160), max(20, y0 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 1.8, red, 3, cv2.LINE_AA)
    return bgr


def _pad(panel):
    return cv2.copyMakeBorder(panel, PAD_PX, PAD_PX, PAD_PX, PAD_PX, cv2.BORDER_CONSTANT, value=(255, 255, 255))


def _run_name():
    return RUN_ID


def render_fragment(name, scroll_id, model=None, c=None):
    mask_full = np.array(Image.open(f"masks/{scroll_id}.png").convert("L"))
    y0, y1, x0, x1 = _mask_bbox(mask_full > 0, MASK_CROP_MARGIN)
    crop_height, crop_width = y1 - y0, x1 - x0
    if DRY_RUN or model is None:
        composite = make_composite(scroll_id)
        if composite.shape != mask_full.shape:
            composite = cv2.resize(composite, (mask_full.shape[1], mask_full.shape[0]), interpolation=cv2.INTER_AREA)
        base = cv2.cvtColor(composite, cv2.COLOR_GRAY2BGR)[y0:y1, x0:x1]
        panels = [base.copy() for _ in range(4)]
        kind = "composite dry run"
    else:
        height, width = mask_full.shape
        raw, tta = predict_map(model, c, scroll_id)
        composite = make_composite(scroll_id)
        if composite.shape != (height, width):
            composite = cv2.resize(composite, (width, height), interpolation=cv2.INTER_AREA)
        composite_panel = cv2.cvtColor(composite, cv2.COLOR_GRAY2BGR)[y0:y1, x0:x1]
        panels = [
            _colorize(raw, (height, width))[y0:y1, x0:x1],
            _colorize(tta, (height, width))[y0:y1, x0:x1],
            composite_panel.copy(), composite_panel.copy(),
        ]
        kind = "raw | tta | composite | scale"
    panels[3] = _red_scale_bar_1cm(panels[3])
    panels = [_pad(panel) for panel in panels]
    big = np.hstack(panels) if crop_height >= crop_width else np.vstack(panels)
    out_dir = os.path.join(OUTPUT_DIR, "test_visualizations", _run_name())
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}.jpg")
    if not cv2.imwrite(out_path, big, [cv2.IMWRITE_JPEG_QUALITY, 92]):
        raise RuntimeError(f"failed to save {out_path}")
    print(f"[saved] {out_path} ({kind})")
    del big, panels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [8]:
# build the exact run architecture and load once
from utils.platform import get_zarr_dir

C = build_config()
C.data.zarr_path = get_zarr_dir()
MODEL = None if DRY_RUN else load_model(C)


def ensure_fragment_mask(scroll_id):
    """refresh the fragment mask from the center layer of its active zarr"""
    zarr_path = os.path.join(C.data.zarr_path, f"{scroll_id}.zarr")
    if not os.path.isdir(zarr_path):
        raise FileNotFoundError(
            f"test zarr is missing from the active zarr directory: {zarr_path}"
        )
    volume = zarr.open(zarr_path, mode="r")
    return load_or_create_midslice_mask(
        volume, os.path.join(REPO, "masks", f"{scroll_id}.png"), refresh=True
    )


print(
    "model:", "skipped" if MODEL is None else C.model.arch,
    "| zarr:", C.data.zarr_path,
    "| context:", C.data.context_size,
    "ds", C.data.context_downsample,
    "| feature_attn_mil:", C.model.feature_attn_mil,
    "| learned_surface:", C.model.learned_surface,
    "| multitile:", f"{C.model.multitile_subtile}x{C.model.multitile_grid}",
)

Model parameters (nnunet3d_lcndz): 5,609,085
loaded /vesuvius/models/archs20/bce_soft_noweight_5090_best_character.pth  params=5,609,085  ignored_training_only=4
model: nnunet3d_lcndz | zarr: /vesuvius/ves_zarrs2 | context: 192 ds 2 | feature_attn_mil: True | learned_surface: True | multitile: 16x4


In [ ]:
# --- test fragment 1: PHerc0813 (updated patch, 33.31cm², 5081×5701) ---
NAME = "auto_grown_20260814140748"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

FileNotFoundError: [Errno 2] No such file or directory: 'masks/20260814140748.png'

In [ ]:
# --- test fragment 2: PHerc0211 merged (5 patches, 7181×6501) ---
NAME = "auto_grown_20260717193517"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# --- test fragment 4: PHerc1447 ---
# 51.27cm^2, 6264×8318 (8.64µm src, upsampled to 9.4µm)
NAME = "auto_grown_20250703034159"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# --- test fragment 3: PHerc1203 ---
# 7.9cm^2, 4035×4455
NAME = "auto_grown_20260720090842"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# Test 5: PHerc0826
NAME = "auto_grown_20260723112922"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# w055 remains a separate holdout and is not part of this test pass
print("[skip] w055 holdout")